# fedOPD - do the claims survive seeds and origins?

Every ranking in this project rests on **seed 0 and one forecast origin**. This notebook re-runs the
three claim-bearing arms - `fedavg` (15 rounds x 2 epochs), `central` (30 pooled epochs, one
optimiser), `local` (20 epochs per trust) - across **5 seeds x 3 origins** and reports spreads. The
origins are 2022-04 and 2023-04 (rolling origins wholly inside the training window - for a model
trained only on months before them, the following year is genuinely unseen) and 2024-04 (the
standard validation origin). No scored month ever touches the sealed test year, which is asserted.

**The claims under test, declared with their pass condition:**

1. *Federation beats going it alone* - `fedavg` MASE below `local`, and `fedavg` up-precision above
   `local`, in sign, across origins (read the mean and sd; a sign that flips across seeds at any
   origin fails).
2. *Pooling does not beat federation* - `central` MASE not below `fedavg` under the same reading.
3. *The up call clears its base rate* - `fedavg` up-precision above that origin's base rate for
   every seed.

This notebook can only weaken results, which is the point. Per-service `fedavg` growth calls are
saved for `16`'s decision-regret evaluation.

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path

CANON = Path("../data/canonical")
panel = pd.read_parquet(CANON / "rtt_panel.parquet")
index = pd.read_csv(CANON / "series_index.csv", parse_dates=["split_month"])

SPLIT = index.split_month.iloc[0]                      # validation start; last legal scored month is SPLIT+11
H, L, HIDDEN, ROUNDS, LOCAL_EPOCHS, BAND = 12, 24, 128, 15, 2, 0.05
ORIGINS = [pd.Timestamp("2022-04-01"), pd.Timestamp("2023-04-01"), SPLIT]
SEEDS = [0, 1, 2, 3, 4]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

wide = panel.pivot(index=["provider_code", "dept"], columns="month", values="demand")
wide = wide.loc[list(zip(index.provider_code, index.dept))]
wide.columns = pd.to_datetime(wide.columns)
trusts = index.provider_code.to_numpy()
eye = np.eye(12, dtype=np.float32)

for o in ORIGINS:                                      # no origin may score into the sealed year
    assert o + pd.DateOffset(months=H - 1) <= SPLIT + pd.DateOffset(months=H - 1), "origin scores into the test year"
print("origins:", ", ".join(o.strftime("%Y-%m") for o in ORIGINS), "| seeds:", SEEDS)

origins: 2022-04, 2023-04, 2024-04 | seeds: [0, 1, 2, 3, 4]


In [2]:
def new_model():
    return nn.Sequential(nn.Linear(L + 12, HIDDEN), nn.ReLU(), nn.Dropout(0.1),
                         nn.Linear(HIDDEN, HIDDEN), nn.ReLU(), nn.Linear(HIDDEN, H)).to(device)

def get_w(m): return [p.detach().clone() for p in m.parameters()]
def set_w(m, w):
    with torch.no_grad():
        for p, q in zip(m.parameters(), w): p.copy_(q)

def train_local(model, X, Y, epochs, lr=1e-3):
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.L1Loss()
    X, Y = X.to(device), Y.to(device)
    for _ in range(epochs):
        perm = torch.randperm(len(X))
        for i in range(0, len(X), 256):
            b = perm[i:i + 256]
            opt.zero_grad(); loss_fn(model(X[b]), Y[b]).backward(); opt.step()

def build(origin):
    # everything an origin needs: client tensors, test inputs, truth, scales - training months only
    train_cols = wide.columns < origin
    score_cols = (wide.columns >= origin) & (wide.columns < origin + pd.DateOffset(months=H))
    assert score_cols.sum() == H
    hist = wide.loc[:, train_cols].to_numpy(dtype=np.float32)
    future = wide.loc[:, score_cols].to_numpy(dtype=np.float32)
    norm = hist.mean(axis=1) + 1.0
    month_ix = np.array([m.month - 1 for m in wide.columns[train_cols]])
    scale = np.abs(hist[:, 12:] - hist[:, :-12]).mean(axis=1)    # in-sample seasonal-naive, this origin
    ref = hist[:, -12:].mean(axis=1)
    truth_growth = future.mean(axis=1) / np.where(ref > 0, ref, np.nan) - 1
    clients = {}
    for code_ in np.unique(trusts):
        rows = np.flatnonzero(trusts == code_)
        xs, ys = [], []
        for r in rows:
            s = hist[r] / norm[r]
            for t in range(len(s) - L - H + 1):
                xs.append(np.concatenate([s[t:t + L], eye[month_ix[t + L - 1]]]))
                ys.append(s[t + L:t + L + H])
        clients[code_] = (torch.tensor(np.array(xs)), torch.tensor(np.array(ys)), rows)
    X_test = torch.tensor(np.concatenate(
        [hist[:, -L:] / norm[:, None], np.repeat(eye[month_ix[-1]][None, :], len(hist), axis=0)], axis=1))
    return clients, X_test, norm, future, scale, ref, truth_growth

def score(pred, ctx):
    _, _, _, future, scale, ref, truth_growth = ctx
    eligible = np.isfinite(truth_growth)
    truth_up = truth_growth > BAND
    mae = np.abs(pred - future).mean(axis=1)
    mase = np.nanmedian(np.where(scale > 0, mae / np.where(scale > 0, scale, 1), np.nan))
    g = pred.mean(axis=1) / np.where(ref > 0, ref, np.nan) - 1
    called = np.isfinite(g) & eligible & (g > BAND)
    prec = truth_up[called].mean() * 100 if called.sum() else np.nan
    return mase, prec, int(called.sum()), truth_up[eligible].mean() * 100, int(eligible.sum()), g

print("build() and score() defined")

build() and score() defined


## The sweep

45 training runs (3 origins x 5 seeds x 3 arms). Each seed re-seeds torch before the arm's model
init, so arms differ only in what the seed is meant to vary - initialisation and batch order.

In [3]:
rows, calls = [], []
for origin in ORIGINS:
    ctx = build(origin)
    clients, X_test, norm, future, scale, ref, truth_growth = ctx
    codes = list(clients)
    sizes = np.array([len(clients[c][0]) for c in codes], dtype=float)
    alpha = sizes / sizes.sum()
    for seed in SEEDS:
        # --- fedavg ---
        torch.manual_seed(seed)
        w = get_w(new_model())
        model = new_model()
        for r in range(ROUNDS):
            acc = [torch.zeros_like(p) for p in w]
            for a, c in zip(alpha, codes):
                set_w(model, w)
                train_local(model, clients[c][0], clients[c][1], LOCAL_EPOCHS)
                for i, p in enumerate(get_w(model)):
                    acc[i] += a * p
            w = acc
            set_w(model, w)
        model.eval()
        with torch.no_grad():
            pred = np.clip(model(X_test.to(device)).cpu().numpy() * norm[:, None], 0, None)
        mase, prec, n_up, base, n_elig, g = score(pred, ctx)
        rows.append({"origin": origin.strftime("%Y-%m"), "seed": seed, "arm": "fedavg",
                     "MASE_median": mase, "precision_up": prec, "n_said_up": n_up,
                     "base_rate": base, "n_eligible": n_elig})
        calls.append(pd.DataFrame({"origin": origin.strftime("%Y-%m"), "seed": seed,
                                   "provider_code": index.provider_code, "dept": index.dept,
                                   "pred_growth": g}))
        # --- central ---
        torch.manual_seed(seed)
        central = new_model()
        X_all = torch.cat([clients[c][0] for c in codes])
        Y_all = torch.cat([clients[c][1] for c in codes])
        train_local(central, X_all, Y_all, epochs=ROUNDS * LOCAL_EPOCHS)
        central.eval()
        with torch.no_grad():
            pred = np.clip(central(X_test.to(device)).cpu().numpy() * norm[:, None], 0, None)
        mase, prec, n_up, _, _, _ = score(pred, ctx)
        rows.append({"origin": origin.strftime("%Y-%m"), "seed": seed, "arm": "central",
                     "MASE_median": mase, "precision_up": prec, "n_said_up": n_up,
                     "base_rate": base, "n_eligible": n_elig})
        # --- local ---
        torch.manual_seed(seed)
        pred = np.zeros_like(future)
        for c in codes:
            m_ = new_model()
            train_local(m_, clients[c][0], clients[c][1], epochs=20)
            m_.eval()
            with torch.no_grad():
                pred[clients[c][2]] = np.clip(
                    m_(X_test[clients[c][2]].to(device)).cpu().numpy() * norm[clients[c][2], None], 0, None)
        mase, prec, n_up, _, _, _ = score(pred, ctx)
        rows.append({"origin": origin.strftime("%Y-%m"), "seed": seed, "arm": "local",
                     "MASE_median": mase, "precision_up": prec, "n_said_up": n_up,
                     "base_rate": base, "n_eligible": n_elig})
        print(f"{origin:%Y-%m} seed {seed}: " + " | ".join(
            f"{r['arm']} {r['MASE_median']:.4f}/{r['precision_up']:.1f}%" for r in rows[-3:]))

res = pd.DataFrame(rows)
res.to_csv(CANON / "robustness.csv", index=False)
pd.concat(calls, ignore_index=True).to_csv(CANON / "origin_calls.csv", index=False)
print("written: robustness.csv, origin_calls.csv")

2022-04 seed 0: fedavg 0.8154/49.3% | central 0.7658/57.2% | local 0.7617/47.6%


2022-04 seed 1: fedavg 0.7583/48.7% | central 0.7092/57.5% | local 0.7563/48.5%


2022-04 seed 2: fedavg 0.7941/50.7% | central 0.7055/59.4% | local 0.7640/52.4%


2022-04 seed 3: fedavg 0.8335/51.3% | central 0.7801/55.8% | local 0.7676/49.8%


2022-04 seed 4: fedavg 0.7476/50.9% | central 0.8264/55.7% | local 0.7670/47.0%


2023-04 seed 0: fedavg 0.4922/56.7% | central 0.6720/52.3% | local 0.5954/43.2%


... (5 outputs left out)

2024-04 seed 1: fedavg 0.5258/67.9% | central 0.5617/65.3% | local 0.5509/55.0%


2024-04 seed 2: fedavg 0.5257/64.2% | central 0.6273/69.6% | local 0.5487/57.3%


2024-04 seed 3: fedavg 0.5313/67.6% | central 0.6972/64.3% | local 0.5547/54.4%


2024-04 seed 4: fedavg 0.5121/68.4% | central 0.5979/68.8% | local 0.5563/56.8%
written: robustness.csv, origin_calls.csv


## The verdicts, mean and sd over seeds

In [4]:
summary = (res.groupby(["origin", "arm"])
              .agg(MASE_mean=("MASE_median", "mean"), MASE_sd=("MASE_median", "std"),
                   prec_mean=("precision_up", "mean"), prec_sd=("precision_up", "std"),
                   base=("base_rate", "first"), n=("n_eligible", "first")))
print(summary.round(4).to_string())
print()

wide_r = res.pivot_table(index=["origin", "seed"], columns="arm",
                         values=["MASE_median", "precision_up"])
d1 = wide_r["MASE_median"]["fedavg"] - wide_r["MASE_median"]["local"]        # claim 1a: negative
d1p = wide_r["precision_up"]["fedavg"] - wide_r["precision_up"]["local"]     # claim 1b: positive
d2 = wide_r["MASE_median"]["central"] - wide_r["MASE_median"]["fedavg"]      # claim 2: non-negative
d3 = res[res.arm == "fedavg"].precision_up - res[res.arm == "fedavg"].base_rate   # claim 3: positive

for name, d, want in [("1a fedavg MASE below local", d1, "negative"),
                      ("1b fedavg up-precision above local", d1p, "positive"),
                      ("2  central MASE not below fedavg", d2, "non-negative"),
                      ("3  fedavg up-precision above base rate", d3, "positive")]:
    vals = d.to_numpy()
    holds = {"negative": (vals < 0).all(), "positive": (vals > 0).all(),
             "non-negative": (vals >= 0).all()}[want]
    print(f"claim {name}: mean {vals.mean():+.4f}, sd {vals.std(ddof=1):.4f}, "
          f"range [{vals.min():+.4f}, {vals.max():+.4f}] over {len(vals)} runs -> "
          f"{'HOLDS in sign everywhere' if holds else 'FLIPS SIGN - not a robust claim'}")
print()
print("base rates and eligible counts are per origin (printed above); precisions read against their")
print("own origin's base rate, never another's.")

                 MASE_mean  MASE_sd  prec_mean  prec_sd     base     n
origin  arm                                                           
2022-04 central     0.7574   0.0509    57.1180   1.5018  51.1839  1985
        fedavg      0.7898   0.0366    50.1961   1.1174  51.1839  1985
        local       0.7633   0.0046    49.0527   2.1212  51.1839  1985
2023-04 central     0.6673   0.0147    52.7894   0.7682  40.7258  1984
        fedavg      0.5008   0.0098    54.3122   2.6100  40.7258  1984
        local       0.5943   0.0038    42.6897   1.1528  40.7258  1984
2024-04 central     0.6241   0.0502    67.0332   2.2169  45.9050  1978
        fedavg      0.5204   0.0102    67.6283   2.1110  45.9050  1978
        local       0.5515   0.0039    56.5721   1.9500  45.9050  1978

claim 1a fedavg MASE below local: mean -0.0327, sd 0.0549, range [-0.1062, +0.0659] over 15 runs -> FLIPS SIGN - not a robust claim
claim 1b fedavg up-precision above local: mean +7.9407, sd 5.5953, range [-1.6816, +15

## What this settles

Which of the project's comparative claims are properties of the problem and which were luck of one
seed and one year. Not settled here: hyperparameter sensitivity (08 covered one axis at a time) and
anything about the sealed test year.